# Structured Output

**Structured Output** means asking an LLM to return its response in a **specific format or structure** instead of plain text.


## Pydantic

**Pydantic allows us to define the expected structure using a Python class.**

**We define the fields and their data types, and the model returns structured data according to that schema.**

### Why Pydantic?

* Provides type validation.
* Defines a clear schema.
* Helps catch invalid data.
* Useful when working with structured application data.

In [2]:
# google gemini specific integration 

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie"),
    year: int = Field(..., description="The release year of the movie"),
    director: str = Field(..., description="The director of the movie"),
    rating: float = Field(..., description="The rating of the movie on a scale from 0 to 10"),

model_with_Structure = model.with_structured_output(Movie)

e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The release year of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The director of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default

In [3]:
# without structured output
model.invoke("Tell me about the movie Avengers Endgame.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': '***Avengers: Endgame*** (2019) is a monumental superhero film produced by Marvel Studios and directed by Anthony and Joe Russo. It serves as the 22nd film in the Marvel Cinematic Universe (MCU) and the direct sequel to 2018’s *Avengers: Infinity War*. \n\nThe movie serves as the epic culmination of the "Infinity Saga," wrapping up storylines that had been developing across 22 films over 11 years.\n\nHere is a breakdown of the movie, its plot, themes, and cultural impact. *(Warning: Spoilers ahead!)*\n\n---\n\n### **The Premise**\nThe film picks up shortly after the events of *Infinity War*, where the cosmic tyrant Thanos successfully collected all six Infinity Stones and used them to snap his fingers, erasing half of all life in the universe (an event known as "The Blip"). \n\n*Endgame* deals with the psychological fallout of this unimaginable loss. The surviving Avengers—including Iron Man, Captain America, Thor, Black Widow, Hulk, and Hawk

In [ ]:
# with structured output
response = model_with_Structure.invoke("Tell me about the movie Avengers Endgame.")
response

Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', rating=8.4)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field
"""A movie model for structured output."""
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie"),
    year: int = Field(..., description="The release year of the movie"),
    director: str = Field(..., description="The director of the movie"),
    rating: float = Field(..., description="The rating of the movie on a scale from 0 to 10"),

model_with_Structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_Structure.invoke("Tell me about the movie Avengers Endgame.")
response

e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The release year of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The director of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Avengers: Endgame",\n  "year": 2019,\n  "director": "Anthony and Joe Russo",\n  "rating": 8.4\n}', 'extras': {'signature': 'EmAKXgFpFH0T1WOYXhKh5OOxbiYCOPKMXjPrKy44OuB98btbD9Ez2SpYusuVlIQ++L4doupeIqKn1QjhC/lecKD5hwzIyF9RluxWzfO8j3GTKw8APAROxWOQ+pK+LYFxmD0='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a105d9-f740-7513-8f9e-08df5b895431-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 44, 'total_tokens': 53, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', rating=8.4),
 'parsing_error': None}

### Nested Structure

In [11]:
#nested structured output
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor"),
    character: str = Field(..., description="The character played by the actor in the movie")
    
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie"),
    year: int = Field,
    actors: list[Actor] = Field(..., description="A list of actors in the movie"),
    genre : list[str] = Field(..., description="A list of genres of the movie")

model_with_nested_structure = model.with_structured_output(Movie)
response = model_with_nested_structure.invoke("Tell me about the movie Avengers Endgame.")
response

e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (<function Field at 0x0000025278C0CF40>,) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of the actor'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:

Movie(title='Avengers: Endgame', year=2019, actors=[Actor(name='Robert Downey Jr.', character='Tony Stark / Iron Man'), Actor(name='Chris Evans', character='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', character='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', character='Thor'), Actor(name='Scarlett Johansson', character='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', character='Clint Barton / Hawkeye')], genre=['Action', 'Adventure', 'Sci-Fi'])

### TypedDict

**TypedDict is another way to define the expected structure of data.**
**Unlike Pydantic, TypedDict mainly provides type hints and structure and does not perform the same runtime validation.**

In [ ]:
from typing_extensions import Annotated , TypedDict
"""Movie model for structured output using TypedDict."""
class Movie(TypedDict):
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,..., "The release year of the movie"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie on a scale from 0 to 10"]

moedl_with_TypedDict_structure = model.with_structured_output(Movie)
response = moedl_with_TypedDict_structure.invoke("Tell me about the movie Avengers Endgame.")
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony and Joe Russo',
 'rating': 8.4}

In [14]:
#nested structured output
from pydantic import BaseModel, Field


class Actor(TypedDict):
    name: str
    character: str 
    
class Movie(TypedDict):
    title: str
    year: int = Field,
    actors: list[Actor]
    genre : list[str] 

model_with_nested_structure = model.with_structured_output(Movie)
response = model_with_nested_structure.invoke("Tell me about the movie Avengers Endgame.")
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'actors': [{'name': 'Robert Downey Jr.',
   'character': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'character': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'character': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'character': 'Thor'},
  {'name': 'Scarlett Johansson',
   'character': 'Natasha Romanoff / Black Widow'}],
 'genre': ['Action', 'Adventure', 'Sci-Fi']}

### With agent 


In [19]:
# with pydantic making an agent with structured output
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """contact information model for structured output"""
    name : str = Field(..., description="The name of the contact"),
    email: str = Field(..., description="The email address of the contact"),
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite"),
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "ContactInfo: Saim Anwar , saim.anwar@example.com , 123-456-7890"}
    ]
})

result["structured_response"]

e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The email address of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-defau

ContactInfo(name='Saim Anwar', email='saim.anwar@example.com', phone='123-456-7890')

# Dataclass

A **dataclass** is a Python class designed to store structured data with less boilerplate code.

It automatically provides useful methods such as an initializer (`__init__`) for the defined fields.

### Example

```python
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int
    city: str

person = Person(
    name="Saim",
    age=22,
    city="Lahore"
)

print(person)

In [21]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo(BaseModel):
    """contact information model for structured output"""
    name : str = Field(..., description="The name of the contact"),
    email: str = Field(..., description="The email address of the contact"),
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite"),
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "ContactInfo: Saim Anwar ,saimanwar@example.com , 123-456-7890"}
    ]
})

result["structured_response"]

e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The email address of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Learning\Generative Ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of the contact'),) is not JSON serializable; excluding default from JSON schema [non-serializable-defau

ContactInfo(name='Saim Anwar', email='saimanwar@example.com', phone='123-456-7890')